In [ ]:
# Load the maternal health dataset
import pandas as pd
import numpy as np

df = pd.read_csv("Maternal Health Risk Data Set.csv")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

# Check whether any feature contains missing values
print("\nMissing values:")
print(df.isnull().sum())


In [ ]:
# Check for duplicate rows
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)

duplicates = df[df.duplicated(keep=False)]

print("Total rows involved in duplicates:", len(duplicates))
print(duplicates.head(20))

# Count unique rows after removing exact duplicates
print("Unique rows:", df.drop_duplicates().shape[0])


In [ ]:
feature_cols = [
    "Age",
    "SystolicBP",
    "DiastolicBP",
    "BS",
    "BodyTemp",
    "HeartRate"
]

label_conflicts = (
    df.groupby(feature_cols)["RiskLevel"]
      .nunique()
)

conflicts = label_conflicts[label_conflicts > 1]

print("Number of feature combinations with conflicting labels:", len(conflicts))

In [ ]:
# Target variable distribution
print(df["RiskLevel"].value_counts())


In [ ]:
# Check the range of numerical features
print(df.nunique())

numerical_columns = [
    "Age",
    "SystolicBP",
    "DiastolicBP",
    "BS",
    "BodyTemp",
    "HeartRate"
]

for column in numerical_columns:
    print(f"{column}: {df[column].min()} - {df[column].max()}")

In [ ]:
import matplotlib.pyplot as plt

# Visualize the distribution of numerical features
df[numerical_columns].hist(figsize=(12, 8), bins=15)

plt.suptitle("Distribution of Numerical Features")
plt.tight_layout()
plt.show()

In [ ]:
# Check correlation between numerical features
correlation_matrix = df[numerical_columns].corr()

print(correlation_matrix)

In [ ]:
# Remove exact duplicate rows from the dataset
df_clean = df.drop_duplicates().copy()

print("Original rows:", len(df))
print("Rows after removing exact duplicates:", len(df_clean))
print("Rows removed:", len(df) - len(df_clean))

In [ ]:
# Check for conflicting labels after removing exact duplicates
conflict_rows = (
    df_clean.groupby(feature_cols)["RiskLevel"]
    .nunique()
)

conflict_features = conflict_rows[conflict_rows > 1]

print("Conflicting feature combinations:", len(conflict_features))
print("\nConflicting combinations:")
print(conflict_features)

In [ ]:
# Identify rows belonging to conflicting feature combinations
conflict_mask = df_clean[feature_cols].apply(tuple, axis=1).isin(
    conflict_features.index
)

print("Conflicting rows:", conflict_mask.sum())
print("Non-conflicting rows:", (~conflict_mask).sum())

In [ ]:
# Create the final cleaned dataset and check its class distribution
df_final = df_clean[~conflict_mask].copy()

print("Final dataset shape:", df_final.shape)
print("Original dataset shape:", df.shape)

print(df_final["RiskLevel"].value_counts())
print("\nPercentage distribution:")
print(df_final["RiskLevel"].value_counts(normalize=True) * 100)

In [ ]:
# Compare average feature values across risk levels in the final dataset
for column in numerical_columns:
    print("\n", column)
    print(df_final.groupby("RiskLevel")[column].mean())

In [ ]:
# Separate features (X) and target variable (y) from the final dataset
X = df_final.drop("RiskLevel", axis=1)
y = df_final["RiskLevel"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nX columns:")
print(X.columns)

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, f1_score, recall_score
from sklearn.preprocessing import LabelEncoder
import numpy as np

# 1. Encode text labels to numbers (0, 1, 2) required by XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y)
high_risk_val = le.transform(['high risk'])[0]

# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print("Training Set Shape:", X_train.shape)
print("Test Set Shape:", X_test.shape)


In [ ]:
# Create Stratified 5-Fold CV
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Define a custom scorer for High-Risk Recall specifically
def high_risk_recall(y_true, y_pred):
    labels = np.unique(y_true)
    if high_risk_val not in labels:
        return 0.0
    recalls = recall_score(y_true, y_pred, average=None, labels=labels, zero_division=0)
    high_risk_idx = list(labels).index(high_risk_val)
    return recalls[high_risk_idx]

# Using lambda wrappers to bypass sklearn's strict signature checks
scoring = {
    'Macro_F1': make_scorer(lambda y_t, y_p: f1_score(y_t, y_p, average='macro', zero_division=0)),
    'Macro_Recall': make_scorer(lambda y_t, y_p: recall_score(y_t, y_p, average='macro', zero_division=0)),
    'High_Risk_Recall': make_scorer(high_risk_recall)
}

def evaluate_baseline(model, name):
    print(f"--- Evaluating {name} ---")
    results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)

    mac_f1 = results['test_Macro_F1'].mean()
    mac_rec = results['test_Macro_Recall'].mean()
    hr_rec = results['test_High_Risk_Recall'].mean()

    print(f"Macro F1-Score:     {mac_f1:.4f}")
    print(f"Macro Recall:       {mac_rec:.4f}")
    print(f"High-Risk Recall:   {hr_rec:.4f}\n")


In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.class_weight import compute_sample_weight

# --- BASELINES ---
dummy = DummyClassifier(strategy='most_frequent')
evaluate_baseline(dummy, "Dummy Classifier (Majority Class)")

gnb = GaussianNB()
evaluate_baseline(gnb, "Gaussian Naive Bayes")

# --- STANDARD MODELS ---
print("=== STANDARD MODELS ===")
logreg_std = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42))
])
evaluate_baseline(logreg_std, "Standard Logistic Regression")

dt_std = DecisionTreeClassifier(random_state=42)
evaluate_baseline(dt_std, "Standard Decision Tree")

rf_std = RandomForestClassifier(random_state=42)
evaluate_baseline(rf_std, "Standard Random Forest")

xgb_std = XGBClassifier(random_state=42, eval_metric='mlogloss')
evaluate_baseline(xgb_std, "Standard XGBoost")

# --- CLASS-WEIGHTED MODELS ---
print("=== CLASS-WEIGHTED MODELS ===")
logreg_bal = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(multi_class='multinomial', class_weight='balanced', solver='lbfgs', max_iter=1000, random_state=42))
])
evaluate_baseline(logreg_bal, "Class-Weighted Logistic Regression")

dt_bal = DecisionTreeClassifier(class_weight='balanced', random_state=42)
evaluate_baseline(dt_bal, "Class-Weighted Decision Tree")

rf_bal = RandomForestClassifier(class_weight='balanced', random_state=42)
evaluate_baseline(rf_bal, "Class-Weighted Random Forest")

class BalancedXGBClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, random_state=42):
        self.random_state = random_state
        self.model = XGBClassifier(random_state=self.random_state, eval_metric='mlogloss')

    def fit(self, X, y):
        weights = compute_sample_weight('balanced', y)
        self.model.fit(X, y, sample_weight=weights)
        self.classes_ = self.model.classes_
        return self

    def predict(self, X):
        return self.model.predict(X)

xgb_bal = BalancedXGBClassifier(random_state=42)
evaluate_baseline(xgb_bal, "Class-Weighted XGBoost")


In [ ]:
# Validation and evaluation setup
# LabelEncoder sorts alphabetically, so the encoded ids are
# high risk = 0, low risk = 1, mid risk = 2.
# A confusion matrix built with default labels therefore comes out in
# high/low/mid order, which makes cm[2] the mid-risk row, not the high-risk one.
# ORDER is passed everywhere below so matrices stay in low < mid < high order.
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, ParameterGrid
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, confusion_matrix)

CLASS_ORDER = ['low risk', 'mid risk', 'high risk']
ORDER = le.transform(CLASS_ORDER)

print("LabelEncoder order:", list(le.classes_))
print("Order used here:   ", CLASS_ORDER, "->", list(ORDER))

In [ ]:
# Cost matrix. Rows are the true class, columns the predicted class, in CLASS_ORDER.
# Correct predictions cost 0; a high-risk patient called low risk costs the most.
# These values are a placeholder and need to be fixed by the cost-sensitive section
# before any result based on cost_per_case goes into the report.

#            pred:  low  mid  high
COST = np.array([[   0,   1,   2],   # true low
                 [   2,   0,   1],   # true mid
                 [   5,   3,   0]],  # true high
                dtype=float)

print(pd.DataFrame(COST, index=['true ' + c for c in CLASS_ORDER],
                         columns=['pred ' + c for c in CLASS_ORDER]))

In [ ]:
# Metric function

def score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=ORDER)
    rec = recall_score(y_true, y_pred, labels=ORDER, average=None, zero_division=0)
    prec = precision_score(y_true, y_pred, labels=ORDER, average=None, zero_division=0)

    return {
        'accuracy':      accuracy_score(y_true, y_pred),
        'macro_f1':      f1_score(y_true, y_pred, labels=ORDER, average='macro',
                                  zero_division=0),
        'macro_recall':  rec.mean(),
        'low_recall':    rec[0],
        'mid_recall':    rec[1],
        'high_recall':   rec[2],
        'high_prec':     prec[2],
        'high_fn':       int(cm[2, 0] + cm[2, 1]),   # high called low or mid
        'high_fn_low':   int(cm[2, 0]),              # high called low
        'high_fp':       int(cm[0, 2] + cm[1, 2]),   # false alarms
        'cost_per_case': float((cm * COST).sum() / cm.sum()),
        'confusion':     cm,
    }

NUMERIC = ['accuracy', 'macro_f1', 'macro_recall',
           'low_recall', 'mid_recall', 'high_recall', 'high_prec',
           'high_fn', 'high_fn_low', 'high_fp', 'cost_per_case']

In [ ]:
# Stratified k-fold cross-validation

def cv_evaluate(model, X, y, k=5, seed=42):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)

    folds, cm_total = [], np.zeros((3, 3), dtype=int)
    for train_idx, val_idx in skf.split(X, y):
        m = clone(model)
        m.fit(X.iloc[train_idx], y[train_idx])
        s = score(y[val_idx], m.predict(X.iloc[val_idx]))
        folds.append(s)
        cm_total += s['confusion']

    out = {}
    for key in NUMERIC:
        vals = np.array([f[key] for f in folds], dtype=float)
        out[key] = vals.mean()
        out[key + '_std'] = vals.std()

    out['confusion_total'] = cm_total
    out['per_fold_macro_f1'] = [round(f['macro_f1'], 3) for f in folds]
    out['per_fold_high_recall'] = [round(f['high_recall'], 3) for f in folds]
    return out

In [ ]:
# Hyperparameter selection. Tries every setting in the grid, cross-validates each,
# returns a table sorted best first. It never touches the test set, and it returns
# the winning setting rather than a trained model.
# Selection runs on macro_f1, not accuracy: the majority-class baseline gets a high
# accuracy while catching zero high-risk patients, so accuracy would favour exactly
# the failure this project is about.
# Every row carries macro_f1 and high_recall with their spreads regardless of which
# one was used to sort, so the table can be re-sorted for the other objective without
# running cross-validation a second time.

def select(make_model, grid, X, y, metric='macro_f1', k=5, seed=42, verbose=True):
    assert metric in ('macro_f1', 'macro_recall', 'high_recall'), \
        "Selection must use macro_f1, macro_recall or high_recall, not accuracy."

    keep = ['macro_f1', 'macro_f1_std', 'macro_recall', 'macro_recall_std',
            'high_recall', 'high_recall_std', 'mid_recall', 'mid_recall_std',
            'high_fn', 'high_fn_low', 'high_fp', 'cost_per_case']

    combos, rows = list(ParameterGrid(grid)), []
    for i, params in enumerate(combos, 1):
        r = cv_evaluate(make_model(**params), X, y, k=k, seed=seed)
        row = {'params': str(params)}
        row.update({key: r[key] for key in keep})
        rows.append(row)
        if verbose:
            print(f"  [{i}/{len(combos)}] {params} -> "
                  f"{metric}={r[metric]:.3f} (+/- {r[metric + '_std']:.3f})")

    table = pd.DataFrame(rows).sort_values(metric, ascending=False).reset_index(drop=True)

    if verbose:
        b = table.iloc[0]
        print(f"\n  Winner on {metric}: {b['params']}")
        print(f"  {metric} = {b[metric]:.3f} (+/- {b[metric + '_std']:.3f}), "
              f"high-risk misses per fold = {b['high_fn']:.1f} "
              f"({b['high_fn_low']:.1f} of them called low)")

    return table

In [ ]:
# Comparison table, one row per model, every mean shown with its spread

def compare(models, X, y, k=5, seed=42):
    rows = {}
    for name, m in models.items():
        r = cv_evaluate(m, X, y, k=k, seed=seed)
        rows[name] = {
            'accuracy':       f"{r['accuracy']:.3f}",
            'macro_F1':       f"{r['macro_f1']:.3f} +/- {r['macro_f1_std']:.3f}",
            'macro_recall':   f"{r['macro_recall']:.3f}",
            'low_rec':        f"{r['low_recall']:.3f}",
            'mid_rec':        f"{r['mid_recall']:.3f} +/- {r['mid_recall_std']:.3f}",
            'high_rec':       f"{r['high_recall']:.3f} +/- {r['high_recall_std']:.3f}",
            'high_FN':        f"{r['high_fn']:.1f}",
            'high_FN_as_low': f"{r['high_fn_low']:.1f}",
            'high_FP':        f"{r['high_fp']:.1f}",
            'cost':           f"{r['cost_per_case']:.3f}",
        }
    return pd.DataFrame(rows).T

In [ ]:
# Compares the gap between two models against the fold-to-fold spread
# If the gap is smaller than the spread, the two models are not distinguishable
# on this amount of data and the report should not rank them

def gap_is_real(model_a, model_b, X, y, metric='macro_f1', k=5, seed=42,
                name_a='A', name_b='B'):
    a = cv_evaluate(model_a, X, y, k=k, seed=seed)
    b = cv_evaluate(model_b, X, y, k=k, seed=seed)
    gap = abs(a[metric] - b[metric])
    noise = max(a[metric + '_std'], b[metric + '_std'])
    return {
        f'{name_a}_{metric}': round(a[metric], 4),
        f'{name_b}_{metric}': round(b[metric], 4),
        'gap': round(gap, 4),
        'fold_std': round(noise, 4),
        'verdict': 'real' if gap > noise else 'WITHIN FOLD-TO-FOLD NOISE',
    }

def show_confusion(model, X, y, k=5, seed=42):
    cm = cv_evaluate(model, X, y, k=k, seed=seed)['confusion_total']
    return pd.DataFrame(cm,
                        index=['true ' + c for c in CLASS_ORDER],
                        columns=['pred ' + c for c in CLASS_ORDER])

In [ ]:
# Some rows have identical vitals but different risk labels. No model can get both
# members of such a pair right, so there is a hard ceiling on accuracy and our results
# should be read against it rather than against 100%
# This runs on df_clean, where the conflicts are still present. On df_final it would
# always return 1.0, because those rows were dropped there.

def label_ceiling(df, feature_cols, target='RiskLevel'):
    u = df.drop_duplicates()
    g = u.groupby(feature_cols, sort=False)[target]
    best = g.apply(lambda s: s.value_counts().iloc[0]).sum()
    return {
        'unique_rows':           len(u),
        'distinct_profiles':     g.ngroups,
        'conflicting_profiles':  int((g.nunique() > 1).sum()),
        'max_possible_accuracy': round(best / len(u), 4),
    }

ceiling = label_ceiling(df_clean, feature_cols)
for k_, v_ in ceiling.items():
    print(f"{k_}: {v_}")

print("\nRows used for modelling:", len(df_final))
print("Rows dropped as conflicting:", len(df_clean) - len(df_final))

In [ ]:
# Model factories select() needs a function that returns a fresh unfitted model
# StandardScaler sits inside a Pipeline so it is refitted on the training folds only;
# scaling the whole dataset first would leak fold information.
# multi_class='multinomial' is not passed: it was removed in scikit-learn 1.7 and
# multinomial is the default now.
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import GaussianNB

def mk_logreg(C=1.0, class_weight=None):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('lr', LogisticRegression(C=C, class_weight=class_weight,
                                  max_iter=2000, random_state=42)),
    ])

def mk_tree(max_depth=None, min_samples_leaf=1, class_weight=None):
    return DecisionTreeClassifier(max_depth=max_depth,
                                  min_samples_leaf=min_samples_leaf,
                                  class_weight=class_weight, random_state=42)

def mk_forest(n_estimators=300, max_depth=None, min_samples_leaf=1,
              class_weight=None):
    return RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                  min_samples_leaf=min_samples_leaf,
                                  class_weight=class_weight,
                                  random_state=42, n_jobs=-1)

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

class WeightedXGB(BaseEstimator, ClassifierMixin):

    def __init__(self, balanced=True, max_depth=6, n_estimators=300,
                 learning_rate=0.3, random_state=42):
        self.balanced = balanced
        self.max_depth = max_depth
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state

    def fit(self, X, y):
        self.model_ = XGBClassifier(max_depth=self.max_depth,
                                    n_estimators=self.n_estimators,
                                    learning_rate=self.learning_rate,
                                    random_state=self.random_state,
                                    eval_metric='mlogloss')
        w = compute_sample_weight('balanced', y) if self.balanced else None
        self.model_.fit(X, y, sample_weight=w)
        self.classes_ = self.model_.classes_
        return self

    def predict(self, X):
        return self.model_.predict(X)

    def predict_proba(self, X):
        return self.model_.predict_proba(X)

def mk_xgb(balanced=True, max_depth=6, n_estimators=300, learning_rate=0.3):
    return WeightedXGB(balanced=balanced, max_depth=max_depth,
                       n_estimators=n_estimators, learning_rate=learning_rate)

In [ ]:
# Hyperparameter grids
GRID_LOGREG = {'C': [0.01, 0.1, 1.0, 10.0],
               'class_weight': [None, 'balanced']}

GRID_TREE = {'max_depth': [3, 5, 8, 12, None],
             'min_samples_leaf': [1, 5, 10],
             'class_weight': [None, 'balanced']}

GRID_FOREST = {'n_estimators': [200, 500],
               'max_depth': [5, 10, None],
               'min_samples_leaf': [1, 5],
               'class_weight': [None, 'balanced']}

GRID_XGB = {'balanced': [False, True],
            'max_depth': [3, 5, 8],
            'n_estimators': [200, 400],
            'learning_rate': [0.1, 0.3]}

FACTORIES = {'logreg': mk_logreg, 'tree': mk_tree,
             'forest': mk_forest, 'xgb': mk_xgb}
GRIDS = {'logreg': GRID_LOGREG, 'tree': GRID_TREE,
         'forest': GRID_FOREST, 'xgb': GRID_XGB}

print("Settings to try:",
      {k: len(list(ParameterGrid(v))) for k, v in GRIDS.items()})

In [ ]:
# Selection on macro-F1
sel_f1, best_f1 = {}, {}

for name in FACTORIES:
    print(f"--- {name} (selecting on macro_f1) ---")
    t = select(FACTORIES[name], GRIDS[name], X_train, y_train, metric='macro_f1')
    sel_f1[name] = t
    best_f1[name] = eval(t.iloc[0]['params'])
    print()

print("Winners on macro_f1:")
for name, p in best_f1.items():
    print(f"  {name}: {p}")

In [ ]:
# Selection on high-risk recall
sel_rec, best_rec = {}, {}

for name, t in sel_f1.items():
    t2 = t.sort_values('high_recall', ascending=False).reset_index(drop=True)
    sel_rec[name] = t2
    best_rec[name] = eval(t2.iloc[0]['params'])
    b = t2.iloc[0]
    print(f"--- {name} (selecting on high_recall) ---")
    print(f"  winner: {b['params']}")
    print(f"  high_recall = {b['high_recall']:.3f} (+/- {b['high_recall_std']:.3f}), "
          f"macro_f1 = {b['macro_f1']:.3f}\n")

In [ ]:
# Where the two objectives pick different settings, that difference is the
# recall against false-alarm trade-off showing up during selection.
for name in FACTORIES:
    if best_f1[name] == best_rec[name]:
        print(f"{name}: same setting under both objectives")
        continue

    f1_row, rec_row = sel_f1[name].iloc[0], sel_rec[name].iloc[0]
    print(f"{name}: different")
    print(f"  macro_f1 picks    {best_f1[name]}")
    print(f"    high_recall {f1_row['high_recall']:.3f}, "
          f"high-risk misses {f1_row['high_fn']:.1f}")
    print(f"  high_recall picks {best_rec[name]}")
    print(f"    high_recall {rec_row['high_recall']:.3f}, "
          f"high-risk misses {rec_row['high_fn']:.1f}, "
          f"macro_f1 {rec_row['macro_f1']:.3f}")
    print()

In [ ]:
# Comparison table. Baselines, untuned models, and the tuned models.
# The majority-class row is the important one: reasonable accuracy, zero high-risk recall.
models = {
    'majority':       DummyClassifier(strategy='most_frequent'),
    'gaussian_nb':    GaussianNB(),
    'logreg_default': mk_logreg(),
    'tree_default':   mk_tree(),
    'forest_default': mk_forest(),
    'xgb_default':    mk_xgb(balanced=False),
    'logreg_tuned':   mk_logreg(**best_f1['logreg']),
    'tree_tuned':     mk_tree(**best_f1['tree']),
    'forest_tuned':   mk_forest(**best_f1['forest']),
    'xgb_tuned':      mk_xgb(**best_f1['xgb']),
}

results = compare(models, X_train, y_train)
print(results.to_string())

In [ ]:
# Check which differences survive the fold-to-fold spread.
checks = [
    ('forest_tuned', 'tree_tuned'),
    ('xgb_tuned',    'forest_tuned'),
    ('forest_tuned', 'logreg_tuned'),
    ('logreg_tuned', 'gaussian_nb'),
    ('forest_tuned', 'forest_default'),
]

for a, b in checks:
    r = gap_is_real(models[a], models[b], X_train, y_train,
                    metric='macro_f1', name_a=a, name_b=b)
    print(f"{a:16s} vs {b:16s}  gap={r['gap']:.4f}  "
          f"std={r['fold_std']:.4f}  ->  {r['verdict']}")

In [ ]:
# Confusion matrices, pooled over all five validation folds.
# The bottom-left cell, true high risk predicted low risk, is the one the error
# analysis is built on.
for name in ['majority', 'logreg_tuned', 'tree_tuned', 'forest_tuned', 'xgb_tuned']:
    print(name)
    print(show_confusion(models[name], X_train, y_train).to_string())
    print()

In [ ]:
# Per-fold scores
for name in ['logreg_tuned', 'tree_tuned', 'forest_tuned', 'xgb_tuned']:
    r = cv_evaluate(models[name], X_train, y_train)
    print(name)
    print("  macro_f1 per fold:   ", r['per_fold_macro_f1'])
    print("  high_recall per fold:", r['per_fold_high_recall'])
    print()

In [ ]:
# Numbers for the progress report
print("DATA")
print("  raw rows:", len(df))
print("  after removing exact duplicates:", len(df_clean))
print("  after dropping label conflicts:", len(df_final))
print("  development / test split:", len(X_train), "/", len(X_test))
print("  mid-risk rows in development:",
      int((y_train == le.transform(['mid risk'])[0]).sum()))
print("  accuracy ceiling from label conflicts:", ceiling['max_possible_accuracy'])

print("\nVALIDATION")
print("  stratified 5-fold cross-validation on the development set")
print("  hyperparameters selected inside the folds, on macro-F1")
print("  test set not used")

print("\nRESULTS")
print(results[['macro_F1', 'high_rec', 'high_FN_as_low', 'cost']].to_string())

print("\nLIMITATIONS TO STATE IN THE REPORT")
print("  label conflicts were identified on the full dataset, before the split")
print("  mid-risk has about 12 validation rows per fold, so its per-class recall")
print("    moves roughly 8 points per patient and is only reported with its spread")
print("  data comes from a single region, which limits generalisation")
print("  implausible values remain: HeartRate minimum 7 bpm, Age minimum 10")